In [ ]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor # 병렬 처리를 위한 라이브러리

In [ ]:
# 월간 데이터 파일명과 소스 데이터 폴더 경로 설정
monthly_data_name = 'monthly_data.csv'
source_folder_path = 'source_data/아파트(매매)_실거래가/'

In [ ]:
# 데이터 전처리 함수 정의
def preprocess_data(file_path):
    # 데이터 불러오기
    tmp = pd.read_csv(
        file_path,
        encoding='cp949',
        dtype='string',
        na_values='-',
        skiprows=15
    )

    # 수도권/비수도권 분류
    tmp['sido_nm'] = [sgg.split(' ')[0] for sgg in tmp['시군구']]
    tmp['type'] = ['수도권' if sido_nm in ('서울특별시','인천광역시','경기도') else '비수도권' for sido_nm in tmp['sido_nm']]

    # 계약년월
    tmp['contract_dt'] = pd.to_datetime(tmp['계약년월']+tmp['계약일'])
    tmp['contract_ym'] = tmp['contract_dt'].dt.to_period('M')

    # 거래금액 정리
    tmp['deal_amount'] = tmp['거래금액(만원)'].str.replace(',', '').astype('int64')*10_000

    # 유효하지 않은 데이터 제거
    tmp = tmp[tmp['해제사유발생일'].isna()]

    return tmp[['sido_nm','type','contract_ym','deal_amount']].groupby(['type','contract_ym']).agg(
        deal_count = ('type','count'),
        amount_median = ('deal_amount','median'),
    ).reset_index()

In [30]:
# 병렬처리를 위한 파일 경로 리스트 생성
file_paths = [source_folder_path + file_name for file_name in os.listdir(source_folder_path) if file_name.endswith('.csv')]

In [ ]:
# 함수 테스트
preprocess_data(file_paths[0])

,type,contract_ym,deal_count,amount_median
0,비수도권,2021-01,29998,178000000.0
1,수도권,2021-01,29502,397000000.0


In [32]:
# 월간 데이터 생성 (병렬처리)
with ThreadPoolExecutor() as executor:
    results = list(executor.map(preprocess_data, file_paths))

In [33]:
# 결과 병합
monthly_data = pd.concat(results, ignore_index=True)

In [34]:
# 결과 확인
monthly_data

,type,contract_ym,deal_count,amount_median
0,비수도권,2021-01,29998,178000000.0
1,수도권,2021-01,29502,397000000.0
2,비수도권,2021-02,26525,170000000.0
3,수도권,2021-02,23654,360000000.0
4,비수도권,2021-03,34161,169000000.0
...,...,...,...,...
91,수도권,2020-10,25015,370000000.0
92,비수도권,2020-11,57638,237000000.0
93,수도권,2020-11,31408,420000000.0
94,비수도권,2020-12,46314,194500000.0


In [ ]:
# 계약년월별로 데이터가 잘 정리되었는지 확인
monthly_data['contract_ym'].astype(str).sort_values().unique()

array(['2020-01', '2020-02', '2020-03', '2020-04', '2020-05', '2020-06',
       '2020-07', '2020-08', '2020-09', '2020-10', '2020-11', '2020-12',
       '2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06',
       '2021-07', '2021-08', '2021-09', '2021-10', '2021-11', '2021-12',
       '2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06',
       '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12',
       '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06',
       '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12'],
      dtype=object)

In [ ]:
# 월간 데이터 저장
monthly_data.to_csv(monthly_data_name, index=False)